# LOCO Validation — Immunogenicity Calibrator (Issue #547)

Goal: validate the KDE-based immunogenicity calibrator (`PresentationCalibrator`) via leave-one-cohort-out (LOCO) cross-validation.

**Key caveat:** none of the cohorts (NCI, TESLA, HiTIDE, IMPROVE) is a splice-junction neoepitope cohort — all four are SNV point-mutation datasets. LOCO therefore tests cross-lab/assay transferability as the best available stand-in, not point-mutation → splice transfer directly. **LOCO success ≠ splice validation.**

Load libraries, data, and true-count table; print per-cohort label distribution.

In [1]:
import sys, os
sys.path.insert(0, "/Users/jin-holee/dev/GitHub/Jin-HoMLee/splice-neoepitope-pipeline-scientist/research/experiments/issue_547_immunogenicity_calibration")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
import warnings
warnings.filterwarnings("ignore")

from calibrator import PresentationCalibrator

BASE = "/Users/jin-holee/dev/GitHub/Jin-HoMLee/splice-neoepitope-pipeline-scientist/research/experiments/issue_547_immunogenicity_calibration"
df = pd.read_parquet(os.path.join(BASE, "outputs/scored_cohort_subsample.parquet"))
tc = pd.read_csv(os.path.join(BASE, "outputs/scored_cohort_subsample.parquet.true_counts.csv"))

# Build true-count dicts
true_pos = tc[tc.label==1].set_index("cohort")["count"].to_dict()
true_neg = tc[tc.label==0].set_index("cohort")["count"].to_dict()

COHORTS = ["NCI", "TESLA", "HiTIDE", "IMPROVE"]
print("Per-cohort counts (subsample):")
print(df.groupby(["cohort","label"]).size().reset_index(name="n").to_string(index=False))
print("\nTrue counts (pre-subsample):")
print(tc.to_string(index=False))

Matplotlib is building the font cache; this may take a moment.


Per-cohort counts (subsample):
 cohort  label     n
 HiTIDE      0   173
 HiTIDE      1    41
IMPROVE      0  1938
IMPROVE      1   467
    NCI      0 47809
    NCI      1   103
  TESLA      0    80
  TESLA      1    34

True counts (pre-subsample):
 cohort  label  count
 HiTIDE      0   1522
 HiTIDE      1     41
IMPROVE      0  17053
IMPROVE      1    467
    NCI      0 420683
    NCI      1    103
  TESLA      0    702
  TESLA      1     34


Explain negative-reweighting: subsample keeps all positives but only samples negatives, so raw positive rate is inflated; we reweight negatives to their true frequency for reliability and AUPRC.

**Reweighting negatives to true frequency.** The subsample keeps *all* positives but subsamples negatives cohort-proportionally. The inflated positive rate in the subsample would make any reliability curve appear miscalibrated even for a perfect calibrator. For each held-out cohort we assign: `weight_neg = true_neg[cohort] / sampled_neg[cohort]` and `weight_pos = 1.0`. These sample weights are used when computing the reliability curve (observed positive rate per log-odds bin) and AUPRC. Ranking metrics (positives-in-top-k) are ordering-based and less sensitive to reweighting, but we note this caveat.

Helper functions: reliability curve with Wilson CIs, monotonicity check, and AUPRC.

In [2]:
def wilson_ci(n_pos_w, n_total_w, z=1.96):
    """Wilson score interval for weighted counts."""
    p = n_pos_w / (n_total_w + 1e-12)
    denom = 1 + z**2 / n_total_w if n_total_w > 0 else 1
    centre = (p + z**2 / (2 * n_total_w)) / denom if n_total_w > 0 else p
    margin = z * np.sqrt(p * (1 - p) / n_total_w + z**2 / (4 * n_total_w**2)) / denom if n_total_w > 0 else 0
    return float(centre - margin), float(centre + margin)

def reliability_curve(log_odds, labels, weights, n_bins=10):
    """Bin predicted log-odds; compute weighted observed positive rate + Wilson CI per bin."""
    edges = np.percentile(log_odds, np.linspace(0, 100, n_bins + 1))
    edges = np.unique(edges)
    bin_ids = np.digitize(log_odds, edges[1:-1])
    results = []
    for b in range(len(edges) - 1):
        mask = bin_ids == b
        if mask.sum() == 0:
            continue
        w_pos = weights[mask][labels[mask] == 1].sum() if (labels[mask] == 1).any() else 0.0
        w_tot = weights[mask].sum()
        lo, hi = wilson_ci(w_pos, w_tot)
        results.append({
            "bin_mid": float(np.median(log_odds[mask])),
            "obs_rate": float(w_pos / (w_tot + 1e-12)),
            "ci_lo": lo, "ci_hi": hi,
            "n": int(mask.sum()),
        })
    return pd.DataFrame(results)

def check_monotone(log_odds, labels, weights, n_bins=10):
    """Return fraction of consecutive bin-pairs where observed rate is non-decreasing."""
    rc = reliability_curve(log_odds, labels, weights, n_bins)
    if len(rc) < 2:
        return 1.0, rc
    rates = rc["obs_rate"].values
    violations = np.sum(np.diff(rates) < -0.02)  # allow tiny rounding
    return 1.0 - violations / (len(rates) - 1), rc

def compute_sample_weights(cohort_col, label_col, true_neg_dict, sampled_neg_dict):
    """Return per-row sample weights (neg reweighted to true frequency, pos=1)."""
    weights = np.ones(len(cohort_col))
    for cohort in true_neg_dict:
        neg_mask = (cohort_col == cohort) & (label_col == 0)
        sampled_n = sampled_neg_dict[cohort]
        true_n = true_neg_dict[cohort]
        weights[neg_mask] = true_n / sampled_n if sampled_n > 0 else 1.0
    return weights

def auprc_weighted(labels, scores, weights):
    """Weighted AUPRC via sklearn average_precision_score with sample_weight."""
    return average_precision_score(labels, scores, sample_weight=weights)

def top_k_recall(labels, scores, weights, k=20):
    """Fraction of true positives (by weight) in top-k scored items."""
    order = np.argsort(scores)[::-1][:k]
    top_pos = weights[order][labels[order] == 1].sum()
    total_pos = weights[labels == 1].sum()
    return float(top_pos / (total_pos + 1e-12))

Run LOCO: for each held-out cohort, fit on the remaining three using true counts, transform held-out scores, and evaluate.

In [3]:
loco_results = []

for held_out in COHORTS:
    train_cohorts = [c for c in COHORTS if c != held_out]
    
    # Training data
    train_mask = df["cohort"].isin(train_cohorts)
    train_df = df[train_mask]
    
    # True counts for training cohorts
    n_pos_train = sum(true_pos[c] for c in train_cohorts)
    n_neg_train = sum(true_neg[c] for c in train_cohorts)
    
    # Fit calibrator
    cal = PresentationCalibrator(kde_mode="adaptive", n_grid=512)
    cal.fit(
        train_df["genotype_presentation_score"].values,
        train_df["label"].values,
        n_pos_true=n_pos_train,
        n_neg_true=n_neg_train,
        fit_cohorts=train_cohorts,
    )
    
    # Held-out data
    held_mask = df["cohort"] == held_out
    held_df = df[held_mask]
    log_odds = cal.transform(held_df["genotype_presentation_score"].values)
    labels_h = held_df["label"].values
    
    # Sample weights for held-out cohort
    sampled_neg_h = {held_out: (held_df["label"] == 0).sum()}
    weights_h = compute_sample_weights(
        held_df["cohort"].values, labels_h, 
        {held_out: true_neg[held_out]}, sampled_neg_h
    )
    
    # Metrics
    mono_frac, rc = check_monotone(log_odds, labels_h, weights_h)
    auprc = auprc_weighted(labels_h, log_odds, weights_h)
    top20 = top_k_recall(labels_h, log_odds, weights_h, k=20)
    
    loco_results.append({
        "held_out": held_out,
        "n_pos": int(labels_h.sum()),
        "n_neg_sampled": int((labels_h == 0).sum()),
        "n_neg_true": true_neg[held_out],
        "auprc": round(auprc, 4),
        "top20_recall": round(top20, 4),
        "monotonicity": round(mono_frac, 3),
        "prior": round(cal.prior_, 4),
        "log_odds": log_odds,
        "labels": labels_h,
        "weights": weights_h,
        "rc": rc,
    })

loco_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ("log_odds","labels","weights","rc")} for r in loco_results])
print("LOCO results (adaptive KDE):")
print(loco_df.to_string(index=False))

LOCO results (adaptive KDE):
held_out  n_pos  n_neg_sampled  n_neg_true  auprc  top20_recall  monotonicity   prior
     NCI    103          47809      420683 0.0271        0.0583         1.000 -3.5714
   TESLA     34             80         702 0.2013        0.4412         0.875 -6.5777
  HiTIDE     41            173        1522 0.0706        0.2439         0.889 -6.5874
 IMPROVE    467           1938       17053 0.0322        0.0171         1.000 -7.7731


Run stratified 5-fold cross-validation within each cohort and compare AUPRC to LOCO to quantify the cohort-shift penalty.

In [4]:
within_results = []

for cohort in COHORTS:
    cdf = df[df["cohort"] == cohort].reset_index(drop=True)
    scores_c = cdf["genotype_presentation_score"].values
    labels_c = cdf["label"].values
    sampled_neg_c = int((labels_c == 0).sum())
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_auprcs = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(scores_c, labels_c)):
        tr_scores = scores_c[train_idx]
        tr_labels = labels_c[train_idx]
        val_scores = scores_c[val_idx]
        val_labels = labels_c[val_idx]
        
        if tr_labels.sum() < 2 or (1 - tr_labels).sum() < 2:
            continue
        if val_labels.sum() == 0:
            continue
        
        # True counts scaled to fold size (proportional split)
        fold_frac = len(train_idx) / len(cdf)
        n_pos_fold = max(2, int(true_pos[cohort] * fold_frac))
        n_neg_fold = max(2, int(true_neg[cohort] * fold_frac))
        
        try:
            cal_w = PresentationCalibrator(kde_mode="adaptive", n_grid=512)
            cal_w.fit(tr_scores, tr_labels, n_pos_true=n_pos_fold, n_neg_true=n_neg_fold, fit_cohorts=[cohort])
            val_lo = cal_w.transform(val_scores)
        except Exception as e:
            print(f"  Fold {fold_idx} for {cohort} failed: {e}")
            continue
        
        # Reweight validation negatives
        val_neg_sampled = int((val_labels == 0).sum())
        val_neg_true = true_neg[cohort] * (1 - fold_frac)
        w_val = np.ones(len(val_labels))
        neg_mask_v = val_labels == 0
        if val_neg_sampled > 0:
            w_val[neg_mask_v] = val_neg_true / val_neg_sampled
        
        fold_auprcs.append(auprc_weighted(val_labels, val_lo, w_val))
    
    within_auprc = float(np.mean(fold_auprcs)) if fold_auprcs else float("nan")
    
    # Get LOCO AUPRC for this cohort
    loco_auprc = loco_df[loco_df["held_out"] == cohort]["auprc"].values[0]
    
    within_results.append({
        "cohort": cohort,
        "within_auprc": round(within_auprc, 4),
        "loco_auprc": round(loco_auprc, 4),
        "shift_gap": round(float(within_auprc - loco_auprc), 4),
        "n_folds": len(fold_auprcs),
    })

within_df = pd.DataFrame(within_results)
print("Within-cohort vs LOCO (shift gap = within - LOCO):")
print(within_df.to_string(index=False))
print("\nMean shift gap:", round(within_df["shift_gap"].mean(), 4))

Within-cohort vs LOCO (shift gap = within - LOCO):
 cohort  within_auprc  loco_auprc  shift_gap  n_folds
    NCI        0.0249      0.0271    -0.0022        5
  TESLA        0.5039      0.2013     0.3026        5
 HiTIDE        0.2014      0.0706     0.1308        5
IMPROVE        0.0386      0.0322     0.0064        5

Mean shift gap: 0.1094


Re-run LOCO with fixed-bandwidth KDE to compare reliability with adaptive KDE.

In [5]:
kde_compare = {"adaptive": loco_results, "fixed": []}

for held_out in COHORTS:
    train_cohorts = [c for c in COHORTS if c != held_out]
    train_mask = df["cohort"].isin(train_cohorts)
    train_df = df[train_mask]
    n_pos_train = sum(true_pos[c] for c in train_cohorts)
    n_neg_train = sum(true_neg[c] for c in train_cohorts)
    
    cal_f = PresentationCalibrator(kde_mode="fixed", n_grid=512)
    cal_f.fit(train_df["genotype_presentation_score"].values, train_df["label"].values,
              n_pos_true=n_pos_train, n_neg_true=n_neg_train, fit_cohorts=train_cohorts)
    
    held_mask = df["cohort"] == held_out
    held_df = df[held_mask]
    log_odds_f = cal_f.transform(held_df["genotype_presentation_score"].values)
    labels_h = held_df["label"].values
    sampled_neg_h = {held_out: (held_df["label"] == 0).sum()}
    weights_h = compute_sample_weights(held_df["cohort"].values, labels_h,
                                        {held_out: true_neg[held_out]}, sampled_neg_h)
    
    mono_frac_f, rc_f = check_monotone(log_odds_f, labels_h, weights_h)
    auprc_f = auprc_weighted(labels_h, log_odds_f, weights_h)
    
    kde_compare["fixed"].append({
        "held_out": held_out,
        "auprc": round(auprc_f, 4),
        "monotonicity": round(mono_frac_f, 3),
        "log_odds": log_odds_f,
        "labels": labels_h,
        "weights": weights_h,
        "rc": rc_f,
    })

# Comparison table
adaptive_auprcs = {r["held_out"]: r["auprc"] for r in kde_compare["adaptive"]}
fixed_auprcs = {r["held_out"]: r["auprc"] for r in kde_compare["fixed"]}

print("KDE mode comparison (LOCO AUPRC):")
kde_comp_df = pd.DataFrame([
    {"cohort": c, "adaptive_auprc": adaptive_auprcs[c], "fixed_auprc": fixed_auprcs[c],
     "delta": round(adaptive_auprcs[c] - fixed_auprcs[c], 4)}
    for c in COHORTS
])
print(kde_comp_df.to_string(index=False))
winner_kde = "adaptive" if kde_comp_df["delta"].mean() >= 0 else "fixed"
print(f"\nWinner: {winner_kde} (mean delta adaptive-fixed = {kde_comp_df['delta'].mean():.4f})")

KDE mode comparison (LOCO AUPRC):
 cohort  adaptive_auprc  fixed_auprc   delta
    NCI          0.0271       0.0271  0.0000
  TESLA          0.2013       0.2482 -0.0469
 HiTIDE          0.0706       0.0733 -0.0027
IMPROVE          0.0322       0.0322  0.0000

Winner: fixed (mean delta adaptive-fixed = -0.0124)


Produce diagnostic figures: PR reliability curves, monotonicity plot, shift-gap bar chart, and KDE mode comparison.

In [6]:
import os
out_dir = os.path.join(BASE, "outputs")

# --- Figure 1: PR reliability curves (LOCO, adaptive) ---
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
for ax, res in zip(axes, loco_results):
    rc = res["rc"]
    ax.errorbar(
        rc["bin_mid"], rc["obs_rate"],
        yerr=[rc["obs_rate"] - rc["ci_lo"], rc["ci_hi"] - rc["obs_rate"]],
        fmt="o-", color="#2c7bb6", capsize=4, label="Observed rate"
    )
    ax.set_xlabel("Predicted log-odds")
    ax.set_ylabel("Observed positive rate (reweighted)")
    ax.set_title(f"LOCO held-out: {res['held_out']}\nAUPRC={res['auprc']:.3f}  Mono={res['monotonicity']:.2f}")
    ax.legend(fontsize=8)
plt.suptitle("Reliability curves — LOCO cross-validation (adaptive KDE)", fontsize=13)
plt.tight_layout()
fig.savefig(os.path.join(out_dir, "pr_reliability.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved pr_reliability.png")

# --- Figure 2: Monotonicity (bar chart) ---
fig, ax = plt.subplots(figsize=(7, 4))
mono_vals = [r["monotonicity"] for r in loco_results]
cohort_labels = [r["held_out"] for r in loco_results]
bars = ax.bar(cohort_labels, mono_vals, color=["#1a9641" if v >= 0.8 else "#fdae61" for v in mono_vals])
ax.set_ylabel("Monotonicity fraction")
ax.set_ylim(0, 1.1)
ax.axhline(0.8, ls="--", color="gray", alpha=0.7, label="0.8 threshold")
ax.set_title("LOCO monotonicity (fraction of non-decreasing consecutive bin pairs)")
ax.legend()
for bar, v in zip(bars, mono_vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.02, f"{v:.2f}", ha="center", va="bottom", fontsize=10)
fig.tight_layout()
fig.savefig(os.path.join(out_dir, "monotonicity.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved monotonicity.png")

# --- Figure 3: Shift gap (within vs LOCO AUPRC) ---
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(within_df))
width = 0.35
b1 = ax.bar(x - width/2, within_df["within_auprc"], width, label="Within-cohort (5-fold)", color="#4575b4")
b2 = ax.bar(x + width/2, within_df["loco_auprc"], width, label="LOCO", color="#d73027")
ax.set_xticks(x)
ax.set_xticklabels(within_df["cohort"])
ax.set_ylabel("AUPRC (reweighted)")
ax.set_title("Cohort-shift penalty: within-cohort vs LOCO AUPRC")
ax.legend()
for i, row in within_df.iterrows():
    gap = row["shift_gap"]
    ax.text(i, max(row["within_auprc"], row["loco_auprc"]) + 0.01, f"\u0394={gap:+.3f}", ha="center", fontsize=9)
fig.tight_layout()
fig.savefig(os.path.join(out_dir, "shift_gap.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved shift_gap.png")

# --- Figure 4: KDE mode comparison (overlay reliability for NCI and IMPROVE) ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, cohort in zip(axes, ["NCI", "IMPROVE"]):
    res_a = next(r for r in kde_compare["adaptive"] if r["held_out"] == cohort)
    res_f = next(r for r in kde_compare["fixed"] if r["held_out"] == cohort)
    rc_a, rc_f = res_a["rc"], res_f["rc"]
    ax.plot(rc_a["bin_mid"], rc_a["obs_rate"], "o-", label=f"Adaptive (AUPRC={res_a['auprc']:.3f})", color="#2c7bb6")
    ax.plot(rc_f["bin_mid"], rc_f["obs_rate"], "s--", label=f"Fixed (AUPRC={res_f['auprc']:.3f})", color="#d7191c")
    ax.set_xlabel("Predicted log-odds")
    ax.set_ylabel("Observed positive rate (reweighted)")
    ax.set_title(f"KDE mode comparison — held-out: {cohort}")
    ax.legend(fontsize=9)
plt.suptitle("Adaptive vs Fixed KDE reliability curves", fontsize=13)
plt.tight_layout()
fig.savefig(os.path.join(out_dir, "kde_compare.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved kde_compare.png")
print(f"\nKDE verdict: {winner_kde} KDE wins (mean AUPRC delta = {kde_comp_df['delta'].mean():.4f}).")

Saved pr_reliability.png
Saved monotonicity.png
Saved shift_gap.png


Saved kde_compare.png

KDE verdict: fixed KDE wins (mean AUPRC delta = -0.0124).


Fit the final calibrator on all four cohorts using full true counts and save the artifact.

In [7]:
all_scores = df["genotype_presentation_score"].values
all_labels = df["label"].values
n_pos_all = sum(true_pos[c] for c in COHORTS)
n_neg_all = sum(true_neg[c] for c in COHORTS)

print(f"Final fit: all cohorts={COHORTS}")
print(f"  true n_pos={n_pos_all}, true n_neg={n_neg_all}")
print(f"  prior = log({n_pos_all}/{n_neg_all}) = {np.log(n_pos_all/n_neg_all):.4f}")

cal_final = PresentationCalibrator(kde_mode=winner_kde, n_grid=512)
cal_final.fit(all_scores, all_labels,
              n_pos_true=n_pos_all, n_neg_true=n_neg_all,
              fit_cohorts=COHORTS)

save_path = os.path.join(BASE, "outputs/calibrator_v1.joblib")
cal_final.save(save_path)
print(f"\nSaved: {save_path}")
print(f"  kde_mode: {cal_final.kde_mode}")
print(f"  prior_: {cal_final.prior_:.4f}")
print(f"  score_range_: {cal_final.score_range_}")
print(f"  n_knots: {len(cal_final.cx_)}")
print(f"  fit_cohorts_: {cal_final.fit_cohorts_}")

# Sanity check: monotone on a grid
grid_check = np.linspace(0, 1, 20)
lo_grid = cal_final.transform(grid_check)
diffs = np.diff(lo_grid)
n_viol = int((diffs < -1e-10).sum())
print(f"\nSanity: {n_viol} monotonicity violations on 20-point [0,1] grid (expected 0)")
print("  log-odds at [0, 0.25, 0.5, 0.75, 1.0]:", cal_final.transform(np.array([0,0.25,0.5,0.75,1.0])).round(4).tolist())

Final fit: all cohorts=['NCI', 'TESLA', 'HiTIDE', 'IMPROVE']
  true n_pos=645, true n_neg=439960
  prior = log(645/439960) = -6.5252

Saved: /Users/jin-holee/dev/GitHub/Jin-HoMLee/splice-neoepitope-pipeline-scientist/research/experiments/issue_547_immunogenicity_calibration/outputs/calibrator_v1.joblib
  kde_mode: fixed
  prior_: -6.5252
  score_range_: (0.002158, 0.999999)
  n_knots: 372
  fit_cohorts_: ['NCI', 'TESLA', 'HiTIDE', 'IMPROVE']

Sanity: 0 monotonicity violations on 20-point [0,1] grid (expected 0)
  log-odds at [0, 0.25, 0.5, 0.75, 1.0]: [-9.3381, -5.58, -5.0837, -4.4374, -3.7879]
